# Explorador interactivo de diseño de recompensa normalizado

Este notebook está pensado como banco de decisión de hiperparámetros. Cada sección empieza mostrando la configuración base del bloque con la misma estructura conceptual de `config_CartPole.yaml`, y luego usa sliders para observar cómo cambian los criterios visuales.

La lógica está concentrada en `reward_design_interactive_methods.py`: un orquestador pequeño, módulos por bloque de recompensa y métodos de visualización. No modifica tu configuración real.


## 0. Carga del explorador

El notebook puede abrirse desde la carpeta raíz del proyecto o desde `Diseño de Recompensa`; por eso busca automáticamente `reward_design_interactive_methods.py`.


In [ ]:
from copy import deepcopy
from pathlib import Path
import pprint
import sys

import numpy as np

CANDIDATE_DIRS = [
    Path.cwd(),
    Path.cwd() / "Diseño de Recompensa",
]
NOTEBOOK_DIR = next(
    (path for path in CANDIDATE_DIRS if (path / "reward_design_interactive_methods.py").exists()),
    Path.cwd(),
)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from reward_design_interactive_methods import (
    build_initial_reward_design_config,
    RewardDesignOrchestrator,
    RewardDesignPlotter,
)

REWARD_DESIGN_CONFIG = build_initial_reward_design_config()
orchestrator = RewardDesignOrchestrator(REWARD_DESIGN_CONFIG)
plotter = RewardDesignPlotter(orchestrator)

def show_config_branch(*keys):
    branch = REWARD_DESIGN_CONFIG
    for key in keys:
        branch = branch[key]
    pprint.pp(branch, sort_dicts=False)

print("Explorador cargado desde:", NOTEBOOK_DIR)
print("Bloques:", list(REWARD_DESIGN_CONFIG["reward_base"]["reward_calculation"].keys()))


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except Exception as exc:
    HAS_WIDGETS = False
    widgets = None
    print("ipywidgets no está disponible en este kernel. Puedes llamar manualmente plotter.*")
    print(exc)

import matplotlib.pyplot as plt


## 1. `metric_processing.normalization`

Antes de ajustar pesos, conviene revisar si `L_e` y `L_edot` tienen una escala comparable. Este gráfico responde si el rango normalizado preserva la banda de estabilización y dónde empieza la saturación.


In [ ]:
show_config_branch("reward_base", "reward_calculation", "metric_processing", "normalization")


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_normalization_ranges,
        pendulum_e_span=widgets.FloatSlider(value=0.35, min=0.05, max=0.80, step=0.01, description="p_e span", continuous_update=False),
        pendulum_edot_span=widgets.FloatSlider(value=1.20, min=0.10, max=2.50, step=0.05, description="p_edot", continuous_update=False),
        cart_e_span=widgets.FloatSlider(value=0.30, min=0.05, max=0.80, step=0.01, description="c_e span", continuous_update=False),
        cart_edot_span=widgets.FloatSlider(value=0.25, min=0.05, max=1.00, step=0.01, description="c_edot", continuous_update=False),
    ))
else:
    plotter.plot_normalization_ranges()


## 2. `principal_reward.weighted_exponential_params` con compuertas

Esta rama permite analizar un diseño de costo exponencial ponderado con gates por feature. El gráfico separa dos preguntas: cómo queda el mapa de costo local y cómo las compuertas cambian los pesos efectivos al variar `L_e`.

Usa esto para decidir si la compuerta está ayudando a priorizar recuperación local o si está apagando demasiado pronto `L_edot`/`L_u`.


In [ ]:
show_config_branch("reward_base", "reward_calculation", "principal_reward", "weighted_exponential_params")
show_config_branch("reward_base", "reward_calculation", "principal_reward", "feature_budget_conservation")


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_weighted_gate_surface,
        w_e=widgets.FloatSlider(value=0.60, min=0.0, max=1.0, step=0.05, description="w_e", continuous_update=False),
        w_edot=widgets.FloatSlider(value=0.35, min=0.0, max=1.0, step=0.05, description="w_edot", continuous_update=False),
        w_u=widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description="w_u", continuous_update=False),
        scale_e=widgets.FloatSlider(value=5.0, min=0.1, max=12.0, step=0.2, description="scale_e", continuous_update=False),
        scale_edot=widgets.FloatSlider(value=2.5, min=0.1, max=10.0, step=0.2, description="scale_edot", continuous_update=False),
        scale_u=widgets.FloatSlider(value=1.0, min=0.1, max=8.0, step=0.1, description="scale_u", continuous_update=False),
        gate_edot_scaled=widgets.FloatSlider(value=0.50, min=0.05, max=1.50, step=0.05, description="g_edot s", continuous_update=False),
        gate_u_scaled=widgets.FloatSlider(value=0.30, min=0.05, max=1.50, step=0.05, description="g_u s", continuous_update=False),
        gate_min=widgets.FloatSlider(value=0.0, min=0.0, max=0.8, step=0.05, description="gate min", continuous_update=False),
        L_u_fixed=widgets.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.05, description="L_u fijo", continuous_update=False),
        budget_conservation=widgets.Checkbox(value=False, description="budget"),
        var_obj=widgets.Dropdown(options=["pendulum_angle", "cart_position"], value="pendulum_angle", description="var"),
    ))
else:
    plotter.plot_weighted_gate_surface()


## 3. `principal_reward.nonlinear_local_cost_params`

Esta es la rama de costo local ya propuesta: una combinación simple de `L_e`, `L_edot` y `L_u` condicionado por `exp(-kappa E_v)`. El mapa permite verificar si `alpha_e`, `effort_weight` y `kappa` producen un criterio local robusto.


In [ ]:
show_config_branch("reward_base", "reward_calculation", "principal_reward", "nonlinear_local_cost_params")


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_local_cost_surface,
        alpha_e=widgets.FloatSlider(value=0.65, min=0.05, max=0.95, step=0.05, description="alpha_e", continuous_update=False),
        effort_weight=widgets.FloatSlider(value=0.10, min=0.00, max=0.60, step=0.02, description="beta_u", continuous_update=False),
        effort_gate_kappa=widgets.FloatSlider(value=4.0, min=0.0, max=10.0, step=0.5, description="kappa", continuous_update=False),
        L_u_fixed=widgets.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.05, description="L_u fijo", continuous_update=False),
        var_obj=widgets.Dropdown(options=["pendulum_angle", "cart_position"], value="pendulum_angle", description="var"),
    ))
else:
    plotter.plot_local_cost_surface()


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_effort_gate,
        alpha_e=widgets.FloatSlider(value=0.65, min=0.05, max=0.95, step=0.05, description="alpha_e", continuous_update=False),
        effort_weight=widgets.FloatSlider(value=0.10, min=0.00, max=0.60, step=0.02, description="beta_u", continuous_update=False),
        effort_gate_kappa=widgets.FloatSlider(value=4.0, min=0.0, max=10.0, step=0.5, description="kappa", continuous_update=False),
        var_obj=widgets.Dropdown(options=["pendulum_angle", "cart_position"], value="pendulum_angle", description="var"),
    ))
else:
    plotter.plot_effort_gate()


## 4. `coordination_reward`: potencial, progreso y composición interna

El bloque coordinativo tiene varias piezas: potencial global, progreso global, asignación de crédito, decaimiento por error local, progreso local y deuda local. Aquí la exploración está dividida para que cada gráfico responda una decisión concreta.


In [ ]:
show_config_branch("reward_base", "reward_calculation", "coordination_reward")


### 4.1 Potencial y progreso global

Este gráfico muestra cómo se forma `Phi` y cómo se normaliza `Delta Phi`. En la lógica normalizada actual, el progreso se acota por la cota analítica del potencial; `progress_clip_max` queda como referencia declarativa/legacy.


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_coordination_potential_and_progress,
        potential_weight_e=widgets.FloatSlider(value=0.65, min=0.0, max=1.0, step=0.05, description="w_phi e", continuous_update=False),
        potential_weight_edot=widgets.FloatSlider(value=0.35, min=0.0, max=1.0, step=0.05, description="w_phi edot", continuous_update=False),
        value_power=widgets.FloatSlider(value=1.0, min=0.5, max=4.0, step=0.25, description="power", continuous_update=False),
        square_terms=widgets.Checkbox(value=False, description="square_terms"),
        progress_clip_max=widgets.FloatSlider(value=0.02, min=0.002, max=0.10, step=0.002, description="clip Phi", continuous_update=False),
    ))
else:
    plotter.plot_coordination_potential_and_progress()


### 4.2 Mezcla global/local/debt y decaimiento por error

El mapa responde cuándo la coordinación neta se vuelve positiva o negativa según progreso local y deuda. La curva lateral compara las tres formas de `local_error_decay`.


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_coordination_component_mixer,
        global_progress_weight=widgets.FloatSlider(value=0.70, min=0.0, max=1.0, step=0.05, description="w_global", continuous_update=False),
        local_progress_weight=widgets.FloatSlider(value=0.20, min=0.0, max=1.0, step=0.05, description="w_local", continuous_update=False),
        local_debt_weight=widgets.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.05, description="w_debt", continuous_update=False),
        global_component_fixed=widgets.FloatSlider(value=0.40, min=0.0, max=1.0, step=0.05, description="global 01", continuous_update=False),
        progress_share_pendulum=widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1, description="p share", continuous_update=False),
        debt_share_pendulum=widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1, description="d share", continuous_update=False),
        condition_mode=widgets.Dropdown(options=["always", "global_progress_positive", "all_local_progress_positive", "global_and_all_local_positive"], value="global_progress_positive", description="cond"),
    ))
else:
    plotter.plot_coordination_component_mixer()


### 4.3 Crédito correctivo

Este mapa muestra cómo se reparte el bonus global entre lazos por contribución correctiva positiva.


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_credit_share,
        epsilon_credit=widgets.FloatLogSlider(value=1.0e-6, base=10, min=-9, max=-2, step=1, description="epsilon", continuous_update=False),
    ))
else:
    plotter.plot_credit_share()


## 5. `reward_config.reward_composition`

Este bloque compone los módulos ya normalizados. La pregunta aquí es cuánto bonus coordinativo puede compensar un mal costo local.


In [ ]:
show_config_branch("reward_base", "reward_config", "reward_composition")


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_progress_and_composition,
        principal_weight=widgets.FloatSlider(value=0.75, min=0.05, max=1.00, step=0.05, description="w_principal", continuous_update=False),
        coordination_weight=widgets.FloatSlider(value=0.25, min=0.00, max=1.00, step=0.05, description="w_coord", continuous_update=False),
        progress_clip_max=widgets.FloatSlider(value=0.02, min=0.002, max=0.10, step=0.002, description="clip ref", continuous_update=False),
    ))
else:
    plotter.plot_progress_and_composition()


## 6. `local_reward_extension`

La extensión local permite añadir tres componentes opcionales: señal marginal contra costo previo/baseline, penalización por mover sin mejorar y sinergia entre error, derivada y esfuerzo. Por ahora se exploran como componentes normalizados, sin activar nada en la config real.


In [ ]:
show_config_branch("reward_base", "reward_calculation", "local_reward_extension")


### 6.1 Marginal baseline y movimiento improductivo

El primer panel muestra cuándo el costo actual mejora respecto al costo previo o baseline. El segundo muestra la zona donde mover tendría penalización por no mejorar.


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_local_extension_marginal,
        previous_cost=widgets.FloatSlider(value=0.45, min=0.0, max=1.0, step=0.02, description="J prev", continuous_update=False),
        baseline_cost=widgets.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.02, description="baseline", continuous_update=False),
        tau=widgets.FloatSlider(value=0.10, min=0.01, max=0.80, step=0.01, description="tau", continuous_update=False),
        mode=widgets.Dropdown(options=["immediate_delta", "smoothed_baseline"], value="immediate_delta", description="mode"),
        moved=widgets.Checkbox(value=True, description="moved"),
    ))
else:
    plotter.plot_local_extension_marginal()


### 6.2 Sinergia error-derivada-esfuerzo

La sinergia debe ser positiva cuando la derivada y el esfuerzo contribuyen a corregir el error, y negativa cuando empujan en contra.


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_local_extension_synergy,
        gamma_e=widgets.FloatSlider(value=1.0, min=0.1, max=6.0, step=0.1, description="gamma_e", continuous_update=False),
        gamma_edot=widgets.FloatSlider(value=1.0, min=0.1, max=6.0, step=0.1, description="gamma_edot", continuous_update=False),
        gamma_u=widgets.FloatSlider(value=1.0, min=0.1, max=6.0, step=0.1, description="gamma_u", continuous_update=False),
        effort_fixed=widgets.FloatSlider(value=0.50, min=-1.0, max=1.0, step=0.05, description="u fijo", continuous_update=False),
        var_obj=widgets.Dropdown(options=["pendulum_angle", "cart_position"], value="pendulum_angle", description="var"),
    ))
else:
    plotter.plot_local_extension_synergy()


### 6.3 Composición interna de la extensión local

Este mapa resume la decisión de pesos entre marginal, movimiento y sinergia. La línea negra indica la frontera entre aporte positivo y negativo.


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_local_extension_composition,
        marginal_weight=widgets.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.05, description="w_marg", continuous_update=False),
        movement_weight=widgets.FloatSlider(value=0.25, min=0.0, max=1.0, step=0.05, description="w_move", continuous_update=False),
        synergy_weight=widgets.FloatSlider(value=0.25, min=0.0, max=1.0, step=0.05, description="w_syn", continuous_update=False),
        moved=widgets.Checkbox(value=True, description="moved"),
    ))
else:
    plotter.plot_local_extension_composition()
